<a href="https://colab.research.google.com/github/allenswdb/TReND-CaMinA/blob/main/notebooks/Kenya26/03_04-Wed1toThu2-AllenTutorial/project_templates/Project-4_Behavioral-Modulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Motivation and Notebook Goals

This project is motivated by Niell and Stryker (2010), *Modulation of Visual Responses by Behavioral State in Mouse Visual Cortex*, & a more recent paper by Christensen & Pillow (2022) *Reduced neural activity but improved coding in rodent higher-order visual cortex during locomotion*. These papers showed that visual responses in mouse V1 depend strongly on the behavioral state of the animal, rather than simply reflecting stimulus driven activity. 

Main highlights from these paper & the field at large:
- Visual responses are typically stronger during locomotion than during stationary periods.
- Behavioral state can change response gain and reliability, not just mean firing rate.
- State dependence can reshape how sensory input is represented across the population.
- State-dependent modulation is often population-wide but heterogeneous across cells.
- Vision in behaving animals is action-coupled, so stimulus processing depends on behavioral context. Think: When you move, the world moves relative to you! How does the brain account for this?

Goal of this notebook:
- Reproduce and extend this core idea in Allen Brain Observatory calcium imaging data during visual stimulation
- Quantify how running relates to neural activity at both single-cell and population levels.
- Compare tuning and response metrics between running and stationary states.

In short: this notebook asks how much "what the animal is doing" changes "what visual cortex is encoding visual information"

In [ ]:
# @title Run to initialize Allen Brain Observatory on Colab {display-mode: "form" }

# run only once per runtime/session, and only if running in colab
# the runtime will need to restxart after
%%capture
!apt install s3fs

!pip uninstall -y numpy pandas
!pip install git+https://github.com/AllenInstitute/AllenSDK@1bdca3ad884c3a5edea8236161424650603e6f29 "numpy == 1.26.4" "pandas == 2.3.0" "matplotlib > 3.8.0" "statsmodels >= 0.14.4"
import allensdk
print('allensdk imported successfully')

!mkdir -p /data/allen-brain-observatory/
!s3fs allen-brain-observatory /data/allen-brain-observatory/ -o public_bucket=1

import time
print("Runtime is now restarting...")
print("You can ignore the error message [Your session crashed for an unknown reason.]")
time.sleep(5)
exit()

In [ ]:
%matplotlib inline

## Import Required Libraries
#Base
import os
from pathlib import Path
import numpy as np
import pandas as pd

#Stat
import scipy.stats as st
from scipy.interpolate import interp1d
from scipy.stats import pearsonr, spearmanr
from scipy.ndimage import gaussian_filter

#Core
from allensdk.core.brain_observatory_cache import BrainObservatoryCache
from allensdk.core.brain_observatory_nwb_data_set import BrainObservatoryNwbDataSet

#Plot
import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.backends.backend_pdf import PdfPages

#User
import util_project_4 as util 
import warnings 
warnings.filterwarnings("ignore")


In [ ]:
import platform, os, sys
platstring = platform.platform()

if ('amzn' in platstring) or ('google.colab' in sys.modules):
    # for AWS
    vc_cache_dir = '/data/allen-brain-observatory/visual-coding-2p'
else:  
    # for local drive, different operating systems
    if ('Darwin' in platstring) or ('macOS' in platstring):
        # OS X 
        data_root = "/Volumes/TReND2026/"
    elif 'Windows'  in platstring:
        # Windows (replace with the drive letter of USB drive)
        data_root = "E:/"
    else:
        # your own linux platform
        # EDIT location where you mounted hard drive
        data_root = "/media/$USERNAME/TReND2026/"
        
    # visual behavior cache directory
    vc_cache_dir = os.path.join(data_root, "visual_coding_2p")
    
boc = BrainObservatoryCache(manifest_file=os.path.join(vc_cache_dir, 'manifest.json'))

In [ ]:
#Meta
orivals = np.arange(0,316,45)      # Unique orientations (degrees)
tfvals = np.array([1,2,4,8,15])    # Unique temporal frequencies (Hz)

#Data
DATA_DIR = Path("/data/visual_coding_ophys")
MANIFEST_PATH = DATA_DIR / "brain_observatory_manifest.json"
NWB_DIR = DATA_DIR / "ophys_experiment_data"

session_types = ['three_session_A','three_session_B','three_session_C','three_session_C2']
# Initialize the cache. Setting `manifest_file` to the local manifest avoids any download.
boc = BrainObservatoryCache(manifest_file=str(MANIFEST_PATH))

# Get high-level metadata tables
containers_df = pd.DataFrame(boc.get_experiment_containers())
containers_df = containers_df.rename(columns={'id':'experiment_container_id'}).set_index('experiment_container_id')
experiments_df = pd.DataFrame(boc.get_ophys_experiments())
experiments_df = experiments_df[experiments_df['session_type'].isin(session_types)]
uniq_container_ids = experiments_df['experiment_container_id'].unique()
containers_df = containers_df.loc[uniq_container_ids]
experiments_df.set_index('id', inplace=True)

print(f"{len(containers_df)} experiment containers, {len(experiments_df)} ophys experiments")
experiments_df.head()

In [ ]:
#Read in pandas dataframe with pre-computed metrics for all cells
#We will use this later
metrics_all = pd.read_csv("metrics_clean_2025.csv",index_col=0)
metrics_dg = metrics_all[['cell_specimen_id','experiment_container_id','tld1_name','area','imaging_depth','pref_ori_dg','pref_tf_dg',
            'num_pref_trials_dg','responsive_dg','g_osi_dg','g_dsi_dg','tfdi_dg','reliability_dg','lifetime_sparseness_dg',
            'fit_tf_dg','fit_tf_ind_dg','tf_low_cutoff_dg','tf_high_cutoff_dg','run_pval_dg','run_resp_dg','stat_resp_dg','run_mod_dg','run_stat_dg',
            'responsive_rundg','run_corr_mean','run_corr_A_lw','run_corr_B_lw','run_corr_C_lw','responsive_run']]
metrics_dg = metrics_dg.set_index('cell_specimen_id')
metrics_dg.head()

# 1. Load an example 2-Photon Session File & plot it's activity

Pick one of the locally available experiments and open its NWB file with `BrainObservatoryNwbDataSet`. This wrapper exposes convenience methods for dF/F, running speed, ROI masks, stimulus tables, and metadata.


In [ ]:
# Pick the first available experiment as our example session
experiment_id = 664404274# 664404274 #501021421 #570305847 #566307038
nwb_path = NWB_DIR / f"{experiment_id}.nwb"
print(f"Loading experiment {experiment_id} from {nwb_path}")

# dataset = BrainObservatoryNwbDataSet(str(nwb_path))
dataset = boc.get_ophys_experiment_data(experiment_id)

#Inspect metadata 
metadata = dataset.get_metadata()
for key, value in metadata.items():
    print(f"{key:>28}: {value}")

print("\nStimuli presented in this session:")
print(dataset.list_stimuli())


**Extract Calcium Activity Traces (dF/F) & running speed**

`get_dff_traces()` returns a tuple `(timestamps, dff)` where `dff` has shape `(n_cells, n_frames)`. Each row corresponds to one ROI (cell). Cell IDs are available via `get_cell_specimen_ids()`.
`get_running_speed()` returns `(dxcm, timestamps)` — running speed in cm/s sampled on the stimulus/imaging clock. NaNs may appear where the encoder signal was unreliable.


In [ ]:
dff_ts, dff = dataset.get_dff_traces() #boc.get_events_traces()
deconvolved_events = boc.get_ophys_experiment_events(experiment_id)
cell_ids = dataset.get_cell_specimen_ids()
frame_rate = 1.0 / np.median(np.diff(dff_ts))
print(f"dF/F shape:        {dff.shape}  (n_cells, n_frames)")
print(f"Events shape:      {deconvolved_events.shape} (n_cells, n_frames)")
print(f"Timestamps shape:  {dff_ts.shape}")
print(f"Frame rate:        {frame_rate:.2f} Hz")
print(f"Session duration:  {dff_ts[-1] - dff_ts[0]:.1f} s")
print(f"Number of cells:   {len(cell_ids)}")


In [ ]:
running_speed, running_ts = dataset.get_running_speed()

print(f"Running speed shape: {running_speed.shape}")
print(f"Running ts shape:    {running_ts.shape}")
print(f"NaN samples:         {np.isnan(running_speed).sum()} / {running_speed.size}")
print(f"Frame rate:          {1.0 / np.median(np.diff(running_ts)):.2f} Hz")
print(f"Mean speed (cm/s):   {np.nanmean(running_speed):.2f}")
print(f"Max speed (cm/s):    {np.nanmax(running_speed):.2f}")


**Visualize running speed & calcium traces during spontaneous periods**

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(running_ts, running_speed, label="Running speed (raw)", alpha=0.5)
# ax.plot(running_ts, running_speed_smooth, label="Running speed (smoothed)", linewidth=2)
ax.set_ylim([np.nanpercentile(running_speed, 1), np.nanpercentile(running_speed, 99)])
ax.set_xlabel("Time (s)")
ax.set_ylabel("Running speed (cm/s)")

In [ ]:
#The start and end columns are indices to the dff & behavior arrays
stim_df = dataset.get_stimulus_epoch_table()
stim_df

In [ ]:
stim = 'spontaneous'  # Change this to the stimulus of interest (e.g., 'drifting_gratings', 'natural_scenes', etc.)

#Find the time slice corresponding to the spontaneous period
start_end_i = stim_df.loc[stim_df.stimulus == stim][['start','end']].values[0]
time_slice = slice(start_end_i[0], start_end_i[1])

n_show = min(10, dff.shape[0])
offset = 3.0  # vertical spacing between traces (in dF/F units)

fig, axes = plt.subplots(2,1,figsize=(12, 6),gridspec_kw={'height_ratios': [1, .2]})
plt.suptitle(f"Experiment {experiment_id} — dF/F traces and running speed during {stim} period", fontsize=16)
ax = axes[0]
for i in range(n_show):
    ax.plot(dff_ts[time_slice]/60, dff[i, time_slice] + i * offset, lw=0.6)
ax.set_xlabel("Time (min)")
ax.set_ylabel("dF/F")
ax.set_title(f"dF/F traces — first {n_show} cells (experiment {experiment_id})")
ax.set_yticks([i * offset for i in range(n_show)])
ax.set_yticklabels([f"cell {i}" for i in range(n_show)])

ax = axes[1]
ax.plot(running_ts[time_slice]/60, running_speed[time_slice], color='orange')
ax.set_xlabel("Time (min)")
ax.set_ylabel("Speed (cm/s)")

for ax in axes:
    ax.autoscale(enable=True, axis='x', tight=True)

<div style="background: #FFF0F0; border-radius: 3px; padding: 10px;">
Let's plot the top 10 highest firing neurons. Can you use the dff to find these units? Plot them during the spontaneous & first drifting grating epochs. What do you notice?
</div>

# 2. Investigate the correlation between single unit activity and running speed


<div style="background: #FFF0F0; border-radius: 3px; padding: 10px;">
<b>Exercise 2a.</b> For each cell, compute the correlation between its dF/F trace and running speed. Hint: To get a good correlation estimate try binning the neural activity and locomotion into larger bins or doing some sort of smoothing procedure (gaussian_filter is easiest), since the frame_rate of 30Hz will lead to low correlations. Try putting your correlation values into a pandas dataframe with the cell_specimen_id as a column as well. 
</div>

<div style="background: #DFF0D8; border-radius: 3px; padding: 10px;">
Compare to correlation values you calculated to those in the pre-computed metrics table to verify things are working properly! Bonus: We're going to learn some pandas magic here
</div>

In [ ]:
#Here we can use the cell_ids to index the giant table of metrics for all cells, and pull out the columns we care about for this analysis
metrics_sub = metrics_dg.loc[cell_ids, ["pref_ori_dg","pref_tf_dg","run_mod_dg",'run_corr_mean','run_corr_A_lw','run_corr_B_lw','run_corr_C_lw', "responsive_dg","responsive_rundg", "responsive_run"]]
metrics_sub.head()

In [ ]:
#Next we can merge the correlation results you just computed with this dataframe by matching on the cell_specimen_id column, which is the unique identifier for each cell and is present in both dataframes
metrics_sub = metrics_sub.merge(corr_df, on="cell_specimen_id", how="left")
metrics_sub.head()

In [ ]:
#Now we have a dataframe with one row per cell, containing both the pre-computed metrics and the new correlation results. We can make scatter plots to compare the new correlations with the pre-computed ones, just to check our work!


<div style="background: #FFF0F0; border-radius: 3px; padding: 10px;">
<b>Exercise 2b.</b> Ok, so we just calculated the correlation across the whole 1hr session. But does the correlation change during different stimulus epochs? Use the stimulus table to index the dff & running traces to subselect different windows of the data. 
</div>

In [ ]:
# Remember that the stimulus epoch table contains the start and end times of each stimulus presentation, as well as the stimulus name.
# We can use this to extract the time periods corresponding to different stimuli and analyze the neural responses during those periods.
stim_df = dataset.get_stimulus_epoch_table()
stim_df

<div style="background: #FFF0F0; border-radius: 3px; padding: 10px;">
<b>Exercise 2c.</b> Plot the distribution of correlations across different stimuli as a histogram. What do you notice about the difference between spontaneous and drifting grating epochs?
</div>

<div style="background: #FFF0F0; border-radius: 3px; padding: 10px;">
<b>Exercise 2d.</b> Let's plot the activity of neurons that are most correlated with behavior during one of the drifting_grating stimulus presentations. There are 3 drifting_grating epochs, so we try looking at the first. 
</div>

# 3. Investigate how locomotion changes the tuning properties of single units

In [ ]:
## FUNCTIONS
##------------------------------------------
def get_dff_traces_and_stim_table(cell_specimen_id, stimulus='drifting_gratings', dataset=None, use_events=True): 
    if dataset is None:
        #identify the session for a given cell id and stimulus
        exps = boc.get_ophys_experiments(cell_specimen_ids=[cell_specimen_id], stimuli=[stimulus])

        #get the experiment_id for that session
        experiment_id = exps[0]['id']
        
        #access the data for that session
        dataset = boc.get_ophys_experiment_data(experiment_id)
    else:
        metadata = dataset.get_metadata()
        experiment_id = metadata['ophys_experiment_id']
        
    #get the DFF or event trace for the cell
    timestamps, dff = dataset.get_dff_traces(cell_specimen_ids=[cell_specimen_id])
    cell_specimen_ids = dataset.get_cell_specimen_ids()
    if use_events:
        deconvolved_events = boc.get_ophys_experiment_events(experiment_id)
        idx = np.where(cell_specimen_ids == cell_specimen_id)[0][0]
        activity_trace = deconvolved_events[idx]
    else:
        activity_trace = dff[0,:]
    
    #get the stimulus table for the stimulus
    stim_table = dataset.get_stimulus_table(stimulus)
    
    return (timestamps, activity_trace, stim_table)

##------------------------------------------
def compute_tuning(cell_specimen_id, stimulus='drifting_gratings', dataset=None, ax=None, plot=True,z_score=True,
                  pref_ori=None,pref_tf=None):
    #get the dff_traces and the stimulus table using the function above
    timestamps, activity_trace, stim_table = get_dff_traces_and_stim_table(cell_specimen_id, stimulus, dataset)
    
    #compute the cell response
    cell_response= np.zeros((len(stim_table),3))
    for i in range(len(stim_table)):
        cell_response[i,0] = stim_table.orientation[i]
        cell_response[i,1] = stim_table.temporal_frequency[i]
        cell_response[i,2] = activity_trace[stim_table.start[i]:stim_table.end[i]].mean()
    
    #get the orivals and tfvals
    all_ori = np.unique(cell_response[:,0])
    orivals = all_ori[np.isfinite(all_ori)]
    tfvals = np.unique(cell_response[:,1])
    tfvals = tfvals[np.isfinite(tfvals)]
    
    #compute the tuning array
    nOri = len(orivals)
    nTF = len(tfvals)
    tuning_array = np.empty((nOri, nTF))
    for i,tf in enumerate(tfvals):
        tf_trials = cell_response[:,1]==tf
        subset = cell_response[tf_trials]
        for j,ori in enumerate(orivals):
            trials = subset[:,0]==ori
            tuning_array[j,i] = subset[trials,2].mean() 
    
    if z_score:
        tuning_array = (tuning_array - tuning_array.mean())/tuning_array.std()

    if plot:
        if ax is None:
            fig, ax = plt.subplots()
        ax.imshow(tuning_array)
        ax.set_xticks(range(nTF))
        ax.set_xticklabels(tfvals)
        ax.set_yticks(range(nOri))
        ax.set_yticklabels(orivals)
        ax.set_xlabel("TF")
        ax.set_ylabel("Direction")
        ax.set_title("Cell "+str(cell_specimen_id))
        if pref_ori is not None:
            ax.text(np.where(tfvals==pref_tf)[0][0],np.where(orivals==pref_ori)[0][0], "*", color="k", fontsize=12, ha='center', va='center')
    
    return (cell_response, tuning_array)


In [ ]:
metrics_exp = metrics_dg.loc[cell_ids]
metrics_exp.head()

In [ ]:
## Calculate mean running speed during each trial of the drifting gratings stimulus, and add it to the stimulus table
##------------------------------------------
def mean_running_speed_per_trial(dataset, stimulus='drifting_gratings', run_thresh = 2):
    speed, ts = dataset.get_running_speed()
    valid = np.isfinite(speed) & np.isfinite(ts)

    speed = speed[valid]
    ts = ts[valid]

    stim_table = dataset.get_stimulus_table(stimulus)

    trial_means = []
    for i, row in stim_table.iterrows():
        start = int(row["start"])
        end = int(row["end"])

        # AllenSDK stimulus tables use frame/sample indices; convert to timestamps if needed
        trial_ts = ts[start:end]
        trial_speed = speed[start:end]
        run_bool = np.nanmean(trial_speed) > run_thresh  # example threshold for "running" vs "stationary"
        trial_means.append({
            "start": start,
            "end": end,
            "mean_running_speed": np.nanmean(trial_speed),
            "run_bool": run_bool
        })
    tmp_df = pd.DataFrame(trial_means)
    stim_table = stim_table.merge(tmp_df, on=["start","end"])

    return stim_table

stim_table_dg = mean_running_speed_per_trial(dataset, "drifting_gratings")
stim_table_dg.head()

In [ ]:
#Let's use the correlation between running and neural activity (run_corr_A_lw) as a feature to predict whether a cell is responsive to the drifting grating stimulus (responsive_dg). We can make a scatter plot of run_corr_A_lw vs responsive_dg, and color by another metric like reliability_dg to see if there are any patterns.
running_sort = np.sort(metrics_exp['run_corr_A_lw'].values)[::-1]
running_idx = np.argsort(metrics_exp['run_corr_A_lw'].values)[::-1]
indy = ~np.isnan(running_sort)
running_sort = running_sort[indy]
running_idx = running_idx[indy]

print(f"Cell indices with highest running correlation: {running_idx[:10]}")


In [ ]:
# Using the compute_tuning function we defined in previous tutorials (now updated to take the dataset as an argument), 
# we can compute the tuning curve for the cell with the highest running correlation, and see if it is responsive to drifting gratings and what its preferred orientation and temporal frequency are.
cell_specimen_id = cell_ids[running_idx[0]]
pref_ori = metrics_dg.loc[cell_specimen_id]['pref_ori_dg']
pref_tf = metrics_dg.loc[cell_specimen_id]['pref_tf_dg']
cell_response, tuning_array = compute_tuning(cell_specimen_id,stimulus='drifting_gratings',dataset=dataset,pref_ori=pref_ori,pref_tf=pref_tf)

In [ ]:
# The heatmap is the most consise way to show the tuning curves for all 5 temporal frequencies across orientations, 
# but we can also plot them as line plots to see the individual curves more clearly and compare the preferred TF to the non-preferred ones.
cmap = sns.color_palette("Set2", len(tfvals))
fig, ax = plt.subplots(figsize=(5, 4))
for i, tf in enumerate(tfvals):
    if tf == pref_tf:
        ax.plot(tuning_array[:,i], label=f"TF = {tfvals[i]}Hz (pref)", marker='o', color=cmap[i],lw=2)
    else:
        ax.plot(tuning_array[:,i], label=f"TF = {tfvals[i]}Hz", marker='o',color=cmap[i],lw=1,alpha=0.5)
ax.set_xticks(range(len(orivals)))
ax.set_xticklabels(orivals)
ax.set_yticks([np.min(tuning_array), np.max(tuning_array)])
ax.set_yticklabels(np.round([np.min(tuning_array), np.max(tuning_array)],1))
ax.set_xlabel("Orientation (degrees)")
ax.set_ylabel("Mean Response (z-scored)",labelpad=-10)
ax.legend(bbox_to_anchor=(1.01, 1))

In [ ]:
##EXERCISE: Let's plot the top 10 cells with the highest running correlation in your preferred way (heatmap, line plot, or both), 
# and see if they are responsive to drifting gratings and what their tuning curves look like. Do you see any patterns in the tuning properties of the cells that are most modulated by running?:



In [ ]:
sub_df = metrics_exp.loc[(metrics_exp.responsive_dg) & (metrics_exp.responsive_rundg)].sort_values('g_osi_dg',ascending=False)

##Let's plot a different subset of neurons: those that are responsive to both the drifting grating stimulus and running, and sort them by their orientation selectivity index (g_osi_dg) 
# to see if there is any relationship between tuning sharpness and running modulation. The OSI metric is a measure of how selective a cell is for its preferred orientation, with higher values indicating sharper tuning. 
# We can see if the cells that are more selective for orientation also tend to be more modulated by running, or if there is no relationship between these features.

    

In [ ]:
## EXERCISE: Let's edit the compute_tuning function to calculate the tuning curves separately for running vs stationary trials,
# and see if the tuning properties of the cell change depending on the animal's behavioral state. We already have a column in the stimulus 
# table that indicates whether each trial was a "running" or "stationary" trial based on the mean running speed during that trial, so we 
# can use that to split the data and compute separate tuning curves for each condition. Some combinations of orientation / temporal frequency may 
# not have enough running or stationary trials to compute a reliable tuning curve, so some tuning curves may be noisy or missing. 
# HINT: we can also "collapse" over temporal frequencies to get more trial per orientation, but this will only give us the preferred orientation, 
# not the preferred temporal frequency, and may miss more subtle changes in tuning that occur at specific TFs.
# Do you see any differences in the preferred orientation, temporal frequency, or tuning sharpness between running and stationary trials for the cells that are modulated by running?

## MORE PROJECT IDEAS & DIRECTIONS OF EXPLORATION

1. Let's compare this experiment that was recorded in VISp with another recorded in the higher visual area VISam 
- I've selected an experiment that has a good balance between resting and running to make comparisons possible. (exp_id = 570305847)
2. Can you think of a way to quantify this modulation of tuning properties due to running? Is it stronger or weaker in VISp or VISam?
- Does this modulation change across natural movies as compared to drifting gratings?
3. Ok, we looked at how single cell activity & tuning properties relates to locomotion. What about population activity?
- How do the latent dynamics of population activity relate to the state of the animal? PCA?

